# 01 · nnU-Net preprocessing (BraTS2023 GLI)

Runs on Kaggle (CPU, internet ON). The mounted dataset `brats2023-gli-nnunet`
has its `.nii.gz` files auto-decompressed by Kaggle into `.nii`, with the **GLI
images nested** inside folders (`X.nii/inner.nii`) and the labels + Africa images
flat (`X.nii`). We rebuild a clean nnU-Net raw tree out of **symlinks** (no copying
of ~165 GB), validate it, then run `nnUNetv2_plan_and_preprocess`.

In [ ]:
# Step 1 — environment: disk, free space, cpu count
import os, subprocess, shutil
print("=== df -h ===")
print(subprocess.run(["df","-h"], capture_output=True, text=True).stdout)
for p in ["/kaggle/working", "/kaggle/tmp", "/tmp"]:
    try:
        u = shutil.disk_usage(p)
        print(f"{p:16s} free {u.free/1e9:6.1f} GB / total {u.total/1e9:6.1f} GB")
    except Exception as e:
        print(f"{p:16s} n/a ({e})")
print("os.cpu_count():", os.cpu_count())
print("nproc:", subprocess.run(["nproc"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Step 2 — install nnunetv2
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nnunetv2"], check=True)
from importlib.metadata import version
print("nnunetv2 version:", version("nnunetv2"))

In [ ]:
# Step 3 — build a symlink raw folder
import os, glob
print("inputs under /kaggle/input:", sorted(glob.glob("/kaggle/input/*")))
MOUNT = "/kaggle/input/brats2023-gli-nnunet"
assert os.path.isdir(MOUNT), f"mount not found: {MOUNT}"
print("mount contents:", sorted(os.listdir(MOUNT)))

RAW = "/kaggle/working/nnUNet_raw/Dataset501_BraTS2023GLI"
FOLDERS = ["imagesTr","labelsTr","imagesTs_gli_test","labelsTs_gli_test","imagesTs_africa","labelsTs_africa"]
for f in FOLDERS:
    os.makedirs(os.path.join(RAW, f), exist_ok=True)

def build_links(folder):
    src_dir, dst_dir = os.path.join(MOUNT, folder), os.path.join(RAW, folder)
    n_file = n_dir = 0
    for entry in sorted(os.listdir(src_dir)):
        src  = os.path.join(src_dir, entry)
        link = os.path.join(dst_dir, entry)      # keep the nnU-Net name X.nii
        if os.path.islink(link) or os.path.exists(link):
            os.remove(link)
        if os.path.isfile(src):                   # flat file (labels, Africa images)
            os.symlink(src, link); n_file += 1
        elif os.path.isdir(src):                  # nested GLI image: X.nii/inner.nii
            inner = os.listdir(src)
            assert len(inner) == 1, f"{src} contains {len(inner)} files: {inner[:3]}"
            os.symlink(os.path.join(src, inner[0]), link); n_dir += 1
    return n_file, n_dir

for f in FOLDERS:
    nf, nd = build_links(f)
    print(f"{f:20s} flat_files={nf:5d}  nested_dirs={nd:5d}")

In [ ]:
# Step 4 — channel check on the nested GLI images
# _0000 -> t1 (NOT t1ce/t1c), _0001 -> t1ce/t1c, _0002 -> t2, _0003 -> flair
from collections import Counter
EXPECT = {"0000": {"t1"}, "0001": {"t1ce","t1c"}, "0002": {"t2"}, "0003": {"flair"}}
def modality_of(inner): return inner.rsplit("_",1)[-1].replace(".nii","").lower()

counts, mismatches = Counter(), []
for folder in ["imagesTr","imagesTs_gli_test"]:
    src_dir = os.path.join(MOUNT, folder)
    for entry in sorted(os.listdir(src_dir)):
        src = os.path.join(src_dir, entry)
        if not os.path.isdir(src):        # flat (Africa) images carry no modality token -> skip
            continue
        chan = entry.split("_")[-1].replace(".nii","")      # e.g. "0000"
        mod  = modality_of(os.listdir(src)[0])
        ok   = mod in EXPECT.get(chan, set())
        counts[(chan, mod, ok)] += 1
        if not ok:
            mismatches.append((folder, entry, chan, mod))

print("channel / modality / ok  ->  count")
for k in sorted(counts):
    print(f"  {k[0]}  {k[1]:6s}  {str(k[2]):5s}  -> {counts[k]}")
if mismatches:
    for m in mismatches[:20]:
        print("MISMATCH:", m)
    raise RuntimeError(f"Channel check FAILED: {len(mismatches)} mismatched images")
print("Channel check PASSED for all nested GLI images.")

In [ ]:
# Step 5 — write dataset.json with file_ending = ".nii"
import json
orig = json.load(open(os.path.join(MOUNT, "dataset.json")))
orig["file_ending"] = ".nii"
json.dump(orig, open(os.path.join(RAW, "dataset.json"), "w"), indent=2)
print(json.dumps(orig, indent=2))

In [ ]:
# Step 6 — verify counts + load 3 random training cases
import nibabel as nib, numpy as np, random
EXPECT_COUNTS = {"imagesTr":4244,"labelsTr":1061,"imagesTs_gli_test":760,
                 "labelsTs_gli_test":190,"imagesTs_africa":380,"labelsTs_africa":95}
for f, exp in EXPECT_COUNTS.items():
    got = len(os.listdir(os.path.join(RAW, f)))
    print(f"{f:20s} {got:5d} (expect {exp}) [{'OK' if got==exp else 'MISMATCH'}]")
    assert got == exp, f"{f}: {got} != {exp}"

cases = sorted(x[:-4] for x in os.listdir(os.path.join(RAW, "labelsTr")))   # strip ".nii"
random.seed(0)
for cid in random.sample(cases, 3):
    shapes = [nib.load(os.path.join(RAW,"imagesTr",f"{cid}_{i:04d}.nii")).shape for i in range(4)]
    seg = nib.load(os.path.join(RAW,"labelsTr",f"{cid}.nii")).get_fdata().astype(int)
    print(cid, "4-channel shapes:", shapes, "| label values:", sorted(np.unique(seg).tolist()))

In [ ]:
# Step 7 — nnU-Net environment variables
os.environ["nnUNet_raw"]          = "/kaggle/working/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/kaggle/working/nnUNet_preprocessed"
os.environ["nnUNet_results"]      = "/kaggle/working/nnUNet_results"
for k in ["nnUNet_raw","nnUNet_preprocessed","nnUNet_results"]:
    os.makedirs(os.environ[k], exist_ok=True)
    print(k, "=", os.environ[k])

In [ ]:
# Step 8 — plan & preprocess (3d_fullres), timed
import time
cmd = ["nnUNetv2_plan_and_preprocess","-d","501","-c","3d_fullres",
       "--verify_dataset_integrity","-np","4"]
print("running:", " ".join(cmd), flush=True)
t0 = time.time()
p = subprocess.run(cmd, env=os.environ, capture_output=True, text=True)
dt = time.time() - t0
print(f"RETURN CODE: {p.returncode}")
print(f"ELAPSED: {dt/60:.1f} min ({dt:.0f} s)")
print("----- STDOUT (tail 80) -----")
print("\n".join(p.stdout.splitlines()[-80:]))
print("----- STDERR (tail 40) -----")
print("\n".join(p.stderr.splitlines()[-40:]))
assert p.returncode == 0, "nnUNetv2_plan_and_preprocess FAILED (see stderr above)"
PP = os.path.join(os.environ["nnUNet_preprocessed"], "Dataset501_BraTS2023GLI")

In [ ]:
# Step 9 — copy splits_final.json into the preprocessed folder
import shutil
os.makedirs(PP, exist_ok=True)
shutil.copy(os.path.join(MOUNT, "splits_final.json"), os.path.join(PP, "splits_final.json"))
print("copied splits_final.json ->", PP)
print(sorted(os.listdir(PP)))

In [ ]:
# Step 10 — print the 3d_fullres plan + preprocessed size
import json
plans = json.load(open(os.path.join(PP, "nnUNetPlans.json")))
cfg   = plans["configurations"]["3d_fullres"]
arch  = cfg.get("architecture", {}); ak = arch.get("arch_kwargs", {})
print("channel_names      :", json.load(open(os.path.join(RAW,"dataset.json")))["channel_names"])
print("patch_size         :", cfg.get("patch_size"))
print("batch_size         :", cfg.get("batch_size"))
print("spacing            :", cfg.get("spacing"))
print("n_stages           :", ak.get("n_stages"))
print("features_per_stage :", ak.get("features_per_stage"))
print("strides            :", ak.get("strides"))
print("normalization      :", cfg.get("normalization_schemes"))
print("network_class      :", arch.get("network_class_name"))
du = subprocess.run(["du","-sh", os.environ["nnUNet_preprocessed"]], capture_output=True, text=True).stdout.strip()
print("nnUNet_preprocessed size:", du)
print(subprocess.run(["du","-sh", PP], capture_output=True, text=True).stdout.strip())